# 02 — Pydantic Validation and Serialization

Field constraints, custom validators, nested models, `Optional`/`Union`, and the aliasing/config knobs that come up whenever an interviewer asks you to design a request/response schema for an API.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*httpx2.*")   # silence a harmless TestClient/httpx notice

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional
from datetime import datetime

app = FastAPI()

## 1. Field constraints

`Field(...)` attaches validation constraints directly to the schema — min/max length, numeric bounds, regex patterns — enforced automatically on every request, and shown in the generated OpenAPI docs so API consumers see the contract without reading code.

In [2]:
class SignupRequest(BaseModel):
    username: str = Field(min_length=3, max_length=20, pattern=r"^[a-zA-Z0-9_]+$")
    age: int = Field(ge=13, le=120)
    referral_code: Optional[str] = Field(default=None, max_length=10)

@app.post("/signup")
def signup(req: SignupRequest):
    return {"accepted": req.model_dump()}

client = TestClient(app)

print(client.post("/signup", json={"username": "al", "age": 25}).status_code)      # 422 -- username too short
print(client.post("/signup", json={"username": "alice", "age": 9}).status_code)     # 422 -- age below ge=13
print(client.post("/signup", json={"username": "alice", "age": 25}).json())          # valid

422
422
{'accepted': {'username': 'alice', 'age': 25, 'referral_code': None}}


## 2. Custom validators

`@field_validator("field_name")` runs custom logic Pydantic's built-in constraints can't express — cross-checking a value's shape, normalizing it, or raising `ValueError` (which Pydantic turns into a structured validation error) for anything that shouldn't reach your handler at all.

In [3]:
class TransactionRequest(BaseModel):
    amount: float
    currency: str

    @field_validator("currency")
    @classmethod
    def currency_must_be_supported(cls, v):
        allowed = {"USD", "EUR", "INR"}
        if v not in allowed:
            raise ValueError(f"currency must be one of {allowed}")
        return v

    @field_validator("amount")
    @classmethod
    def amount_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("amount must be positive")
        return v

@app.post("/transactions")
def create_transaction(tx: TransactionRequest):
    return tx.model_dump()

client = TestClient(app)
print(client.post("/transactions", json={"amount": 50, "currency": "GBP"}).json())   # rejected: unsupported currency
print(client.post("/transactions", json={"amount": -5, "currency": "USD"}).json())    # rejected: not positive
print(client.post("/transactions", json={"amount": 50, "currency": "USD"}).json())    # accepted

{'detail': [{'type': 'value_error', 'loc': ['body', 'currency'], 'msg': "Value error, currency must be one of {'USD', 'EUR', 'INR'}", 'input': 'GBP', 'ctx': {'error': {}}}]}
{'detail': [{'type': 'value_error', 'loc': ['body', 'amount'], 'msg': 'Value error, amount must be positive', 'input': -5, 'ctx': {'error': {}}}]}
{'amount': 50.0, 'currency': 'USD'}


## 3. Nested models and lists

Models compose naturally — a field typed as another `BaseModel` (or `list[SomeModel]`) validates the nested structure recursively, which is how you express a realistic request body like an order with line items.

In [4]:
class LineItem(BaseModel):
    sku: str
    quantity: int = Field(gt=0)
    unit_price: float = Field(gt=0)

class Order(BaseModel):
    customer_id: int
    items: list[LineItem]

    @property
    def total(self) -> float:
        return sum(i.quantity * i.unit_price for i in self.items)

@app.post("/orders")
def create_order(order: Order):
    return {"customer_id": order.customer_id, "total": order.total, "n_items": len(order.items)}

client = TestClient(app)
payload = {
    "customer_id": 1,
    "items": [{"sku": "A1", "quantity": 2, "unit_price": 10.0}, {"sku": "B2", "quantity": 1, "unit_price": 25.0}],
}
print(client.post("/orders", json=payload).json())

bad_payload = {"customer_id": 1, "items": [{"sku": "A1", "quantity": -1, "unit_price": 10.0}]}
print(client.post("/orders", json=bad_payload).status_code)   # 422 -- nested quantity fails gt=0

{'customer_id': 1, 'total': 45.0, 'n_items': 2}
422


## 4. `model_validate` outside a request — using Pydantic standalone

Pydantic models are useful beyond FastAPI request bodies — a common data-engineering pattern is validating rows read from a CSV/JSON file or a Kafka message against a schema *before* they enter a pipeline, catching malformed records early with a clear error instead of an obscure downstream `KeyError`.

In [5]:
raw_records = [
    {"sku": "A1", "quantity": 2, "unit_price": 10.0},
    {"sku": "B2", "quantity": "not-a-number", "unit_price": 5.0},   # malformed
]

valid, invalid = [], []
for record in raw_records:
    try:
        valid.append(LineItem.model_validate(record))
    except ValidationError as e:
        invalid.append((record, str(e)))

print("valid:", valid)
print("invalid count:", len(invalid))

valid: [LineItem(sku='A1', quantity=2, unit_price=10.0)]
invalid count: 1


## 5. Config: extra fields, aliasing

- `model_config = {"extra": "forbid"}` — reject requests containing fields not in the schema (default is to silently ignore them, which can hide client-side typos).
- `Field(alias="...")` — accept/emit a different wire name than the Python attribute name (e.g. camelCase JSON in, snake_case Python attributes internally) — common when a frontend and backend disagree on naming convention.

In [6]:
class StrictItem(BaseModel):
    model_config = {"extra": "forbid"}
    name: str
    price: float

@app.post("/strict-items")
def create_strict_item(item: StrictItem):
    return item.model_dump()

client = TestClient(app)
print(client.post("/strict-items", json={"name": "widget", "price": 9.99, "extra_field": "oops"}).status_code)
# 422 -- "oops" is unexpected, caught immediately instead of being silently dropped

422


## 6. Interview Q&A

1. **"How do you enforce that a field is one of a fixed set of values?"** — an `Enum` type annotation (cleanest, self-documenting in the OpenAPI schema), or a `@field_validator` checking membership against a set for more dynamic cases.
2. **"What happens by default if a client sends extra, unexpected fields?"** — Pydantic silently ignores them unless `model_config = {"extra": "forbid"}` is set — worth calling out as a real footgun (a typo'd field name fails silently instead of erroring).
3. **"How would you validate a batch of records from a file before loading them into a pipeline?"** — loop over records, call `Model.model_validate(record)` in a try/except `ValidationError`, routing failures to a dead-letter list/table instead of crashing the whole batch on one bad row.
4. **"Why use `Field(alias=...)` instead of just renaming the Python attribute?"** — lets Python code use idiomatic `snake_case` internally while still accepting/emitting whatever wire format (often `camelCase`) an external API or frontend requires.

## Summary

- `Field(...)` constraints and `@field_validator` enforce a request contract automatically, before your handler code runs.
- Models nest and validate recursively — the natural way to express structured request bodies like orders with line items.
- Pydantic is useful standalone, not just for FastAPI request bodies — a solid pattern for validating pipeline input records.
- `extra: "forbid"` catches unexpected fields that would otherwise be silently dropped.
- Next: `03_dependency_injection_and_middleware.ipynb`.